# Huấn luyện DeepVQE gốc trên Kaggle (Pure PyTorch) - V2
Notebook này huấn luyện mô hình DeepVQE nguyên bản với bộ VCTK-DEMAND trên Kaggle.

V2 giữ cấu trúc Kaggle gốc, nhưng thêm AMP an toàn, gradient accumulation, DataLoader tối ưu hơn, sqrt-Hann STFT window, checkpoint scaler, log CSV, TensorBoard tùy chọn và PESQ/STOI tùy chọn.

**Không cần SpeechBrain** - toàn bộ logic training viết bằng Pure PyTorch để đảm bảo tương thích mọi môi trường.


## 1. Cài đặt & Thiết lập

In [ ]:
# Kaggle thường đã có torch/torchaudio; tránh pip install torchaudio trần vì có thể kéo lệch version torch.
!pip install tensorboard soundfile pandas tqdm einops pesq pystoi

import os
WORK_DIR = '/kaggle/working/DeepVQE_Workspace'
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
print(f'Thư mục làm việc: {os.getcwd()}')


## 2. Clone mã nguồn DeepVQE

In [ ]:
GIT_REPO = 'https://github.com/hoxuanphu/Pdeepvqe.git'
REPO_DIR = 'deepvqe'

if not os.path.exists(REPO_DIR):
    !git clone {GIT_REPO} {REPO_DIR}
else:
    print(f'Thư mục {REPO_DIR} đã tồn tại.')
    if not os.path.exists(os.path.join(REPO_DIR, 'ablation')):
        print('Lưu ý: repo hiện có chưa có ablation/; cell benchmark sẽ tự lấy từ Pdeepvqe nếu cần.')

os.chdir(REPO_DIR)
print(f'Đã vào thư mục mã nguồn: {os.getcwd()}')

## 3. Quản lý Dataset
Ưu tiên đọc từ `/kaggle/input/` (dataset đính kèm). Nếu không có thì tải về.

In [ ]:
import os
import subprocess
import zipfile
from pathlib import Path

metadata_dir = '/kaggle/working/DeepVQE_Workspace/metadata'
os.makedirs(metadata_dir, exist_ok=True)

required_folders = [
    'clean_trainset_28spk_wav',
    'noisy_trainset_28spk_wav',
    'clean_testset_wav',
    'noisy_testset_wav',
]

datasets_urls = {
    'clean_trainset_28spk_wav': 'https://datashare.ed.ac.uk/bitstream/handle/10283/2791/clean_trainset_28spk_wav.zip?isAllowed=y',
    'noisy_trainset_28spk_wav': 'https://datashare.ed.ac.uk/bitstream/handle/10283/2791/noisy_trainset_28spk_wav.zip?isAllowed=y',
    'clean_testset_wav': 'https://datashare.ed.ac.uk/bitstream/handle/10283/2791/clean_testset_wav.zip?isAllowed=y',
    'noisy_testset_wav': 'https://datashare.ed.ac.uk/bitstream/handle/10283/2791/noisy_testset_wav.zip?isAllowed=y',
}

kaggle_input_root = Path('/kaggle/input')
work_data_dir = Path('/kaggle/working/DeepVQE_Workspace/data/voicebank-demand')

def folder_has_wavs(path):
    path = Path(path)
    return path.is_dir() and any(path.rglob('*.wav'))

def has_required_folders(base_dir):
    base_dir = Path(base_dir)
    return all(folder_has_wavs(base_dir / folder) for folder in required_folders)

def find_attached_dataset():
    if not kaggle_input_root.exists():
        return None
    candidates = [kaggle_input_root / 'voicebank-demand']
    for folder in required_folders:
        candidates.extend(path.parent for path in kaggle_input_root.rglob(folder) if path.is_dir())

    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if has_required_folders(candidate):
            return candidate
    return None

def extract_attached_zips(output_dir):
    if not kaggle_input_root.exists():
        return False
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    extracted_any = False
    for folder_name in required_folders:
        if folder_has_wavs(output_dir / folder_name):
            continue
        matches = list(kaggle_input_root.rglob(folder_name + '.zip'))
        if not matches:
            continue
        zip_path = matches[0]
        print(f'Đang giải nén Kaggle Input zip: {zip_path}')
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(output_dir)
        extracted_any = True
    return extracted_any and has_required_folders(output_dir)

attached_dataset = find_attached_dataset()
if attached_dataset is not None:
    data_dir = str(attached_dataset)
    print(f'Đã tìm thấy dataset đã giải nén ở Kaggle Input: {data_dir}')
elif extract_attached_zips(work_data_dir):
    data_dir = str(work_data_dir)
    print(f'Đã giải nén dataset từ Kaggle Input zip sang: {data_dir}')
else:
    print('Không tìm thấy đủ dataset trong /kaggle/input, sẽ tải từ Edinburgh DataShare...')
    data_dir = str(work_data_dir)
    work_data_dir.mkdir(parents=True, exist_ok=True)

    for folder_name, url in datasets_urls.items():
        extract_path = work_data_dir / folder_name
        zip_path = work_data_dir / f'{folder_name}.zip'

        if folder_has_wavs(extract_path):
            print(f'{folder_name} đã tồn tại, bỏ qua.')
            continue

        if not zip_path.exists() or zip_path.stat().st_size == 0:
            print(f'Đang tải {folder_name}.zip...')
            result = subprocess.run(['wget', '-c', '-O', str(zip_path), url])
            if result.returncode != 0:
                raise RuntimeError(
                    f'Không tải được {folder_name}.zip. '
                    'Hãy bật Internet trong Kaggle Notebook hoặc attach dataset VoiceBank-DEMAND vào /kaggle/input.'
                )

        if not zip_path.exists() or zip_path.stat().st_size == 0:
            raise FileNotFoundError(f'Tải xong nhưng không thấy file: {zip_path}')
        if not zipfile.is_zipfile(zip_path):
            raise zipfile.BadZipFile(f'File tải về không phải zip hợp lệ: {zip_path}')

        print(f'Đang giải nén {folder_name}.zip...')
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(work_data_dir)
        zip_path.unlink()

        if not folder_has_wavs(extract_path):
            raise RuntimeError(f'Đã giải nén nhưng không thấy file wav trong {extract_path}')
        print(f'OK: {folder_name}')

print(f'\nData dir: {data_dir}')
print(f'Metadata dir: {metadata_dir}')

## 4. Tạo file CSV metadata

In [ ]:
import glob
import os
from pathlib import Path

import pandas as pd

def list_wavs(folder):
    return sorted(glob.glob(os.path.join(folder, '**', '*.wav'), recursive=True))

def create_csv(clean_dir, noisy_dir, output_csv):
    clean_files = list_wavs(clean_dir)
    noisy_files = list_wavs(noisy_dir)
    assert len(clean_files) > 0, f'Không tìm thấy file wav trong {clean_dir}'
    assert len(noisy_files) > 0, f'Không tìm thấy file wav trong {noisy_dir}'

    clean_by_id = {Path(path).stem: os.path.abspath(path) for path in clean_files}
    noisy_by_id = {Path(path).stem: os.path.abspath(path) for path in noisy_files}
    ids = sorted(set(clean_by_id) & set(noisy_by_id))
    missing_clean = sorted(set(noisy_by_id) - set(clean_by_id))[:5]
    missing_noisy = sorted(set(clean_by_id) - set(noisy_by_id))[:5]

    assert len(ids) > 0, f'Không tìm thấy cặp clean/noisy trùng ID giữa {clean_dir} và {noisy_dir}'
    assert len(ids) == len(clean_by_id) == len(noisy_by_id), (
        f'Số lượng file không khớp! clean={len(clean_by_id)}, noisy={len(noisy_by_id)}, matched={len(ids)}. '
        f'Missing clean ví dụ: {missing_clean}; missing noisy ví dụ: {missing_noisy}'
    )

    df = pd.DataFrame({
        'ID': ids,
        'clean_wav': [clean_by_id[item_id] for item_id in ids],
        'noisy_wav': [noisy_by_id[item_id] for item_id in ids],
    })
    df.to_csv(output_csv, index=False)
    print(f'Đã tạo {output_csv} với {len(df)} mẫu.')

create_csv(f'{data_dir}/clean_trainset_28spk_wav', f'{data_dir}/noisy_trainset_28spk_wav', f'{metadata_dir}/train_full.csv')
create_csv(f'{data_dir}/clean_testset_wav', f'{data_dir}/noisy_testset_wav', f'{metadata_dir}/test.csv')

df_full = pd.read_csv(f'{metadata_dir}/train_full.csv')
df_full = df_full.sample(frac=1, random_state=42).reset_index(drop=True)
split_idx = int(len(df_full) * 0.90)
df_full.iloc[:split_idx].to_csv(f'{metadata_dir}/train.csv', index=False)
df_full.iloc[split_idx:].to_csv(f'{metadata_dir}/valid.csv', index=False)
print(f'Train: {split_idx} | Valid: {len(df_full) - split_idx}')

## 5. Cấu hình Hyperparameters

In [ ]:
CONFIG = {
    'seed': 1234,
    'sample_rate': 16000,
    'n_fft': 512,
    'hop_length': 256,
    'win_length': 512,
    'stft_window': 'sqrt_hann',   # DeepVQE setup: use the same window for STFT/ISTFT/inference
    
    'batch_size': 4,              # Fast default: match Colab v2 baseline
    'valid_batch_size': 4,
    'grad_accum_steps': 1,        # Increase to 2-4 only if OOM
    'num_workers': 2,
    'persistent_workers': True,
    'prefetch_factor': 2,
    'progress_update_every': 10,  # Reduce notebook UI overhead
    'epochs': 100,
    'lr': 1e-3,
    'grad_clip': 5.0,
    
    # Loss weights
    'lamda_ri': 30,
    'lamda_mag': 70,
    'compress_factor': 0.3,
    
    'train_csv': f'{metadata_dir}/train.csv',
    'valid_csv': f'{metadata_dir}/valid.csv',
    'test_csv': f'{metadata_dir}/test.csv',
    
    'output_dir': '/kaggle/working/DeepVQE_Workspace/checkpoints/deepvqe_vctk',
    
    # Scheduler
    'scheduler_factor': 0.5,
    'scheduler_patience': 5,
    'scheduler_min_lr': 1e-6,
    
    # Early stopping
    'early_stopping_patience': 15,
    
    # V2: Mixed Precision
    'use_amp': True,
    
    # V2: Data Augmentation
    'augment': False,                 # Enable for final-quality runs; keep off for speed baseline
    'aug_gain_range_db': (-6, 6),
    'aug_snr_remix_range': (0, 20),
    'aug_prob': 0.5,
    
    # V2: Optional metrics/logging
    'eval_pesq_every': 0,              # Tính PESQ/STOI mỗi N epoch (0 = không tính)
    'tensorboard': False,              # Enable only when you need live TensorBoard logging
}

print('Config:', CONFIG)


## 6. Dataset & DataLoader (Pure PyTorch)

In [ ]:
import random
import numpy as np
import torch
import torchaudio
from torch.utils.data import Dataset, DataLoader

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(CONFIG['seed'])


class VCTKDemandDataset(Dataset):
    """Dataset cho VCTK-DEMAND: đọc cặp (noisy, clean) từ CSV.
    V2 hỗ trợ data augmentation cho training."""
    
    def __init__(self, csv_path, sample_rate=16000, segment_len=None, augment_cfg=None):
        self.df = pd.read_csv(csv_path)
        self.sample_rate = sample_rate
        self.segment_len = segment_len
        self.aug = augment_cfg
    
    def __len__(self):
        return len(self.df)
    
    def _load(self, path):
        wav, sr = torchaudio.load(path)
        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)
        if sr != self.sample_rate:
            wav = torchaudio.functional.resample(wav, sr, self.sample_rate)
        return wav.squeeze(0)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        noisy = self._load(row['noisy_wav'])
        clean = self._load(row['clean_wav'])
        
        min_len = min(noisy.shape[0], clean.shape[0])
        noisy = noisy[:min_len]
        clean = clean[:min_len]
        
        if self.segment_len is not None and min_len > self.segment_len:
            start = random.randint(0, min_len - self.segment_len)
            noisy = noisy[start:start + self.segment_len]
            clean = clean[start:start + self.segment_len]
        
        if self.aug is not None:
            aug_prob = self.aug.get('aug_prob', 0.5)
            
            if random.random() < aug_prob:
                lo, hi = self.aug['aug_gain_range_db']
                gain_db = random.uniform(lo, hi)
                gain = 10 ** (gain_db / 20)
                noisy = noisy * gain
                clean = clean * gain
            
            if random.random() < aug_prob:
                noise = noisy - clean
                lo, hi = self.aug['aug_snr_remix_range']
                target_snr = random.uniform(lo, hi)
                clean_rms = clean.pow(2).mean().sqrt().clamp(min=1e-8)
                noise_rms = noise.pow(2).mean().sqrt().clamp(min=1e-8)
                target_noise_rms = clean_rms / (10 ** (target_snr / 20))
                noisy = clean + noise * (target_noise_rms / noise_rms)
            
        return {'noisy': noisy, 'clean': clean, 'id': row['ID']}


def collate_fn(batch):
    """Pad tất cả wav trong batch về cùng chiều dài (max len)."""
    max_len = max(item['noisy'].shape[0] for item in batch)
    
    noisy_batch = []
    clean_batch = []
    ids = []
    for item in batch:
        n, c = item['noisy'], item['clean']
        pad_len = max_len - n.shape[0]
        if pad_len > 0:
            n = torch.nn.functional.pad(n, (0, pad_len))
            c = torch.nn.functional.pad(c, (0, pad_len))
        noisy_batch.append(n)
        clean_batch.append(c)
        ids.append(item['id'])
    
    return {
        'noisy': torch.stack(noisy_batch),
        'clean': torch.stack(clean_batch),
        'id': ids,
    }


aug_cfg = {k: v for k, v in CONFIG.items() if k.startswith('aug_')} if CONFIG.get('augment') else None

train_dataset = VCTKDemandDataset(CONFIG['train_csv'], CONFIG['sample_rate'], segment_len=48000, augment_cfg=aug_cfg)
valid_dataset = VCTKDemandDataset(CONFIG['valid_csv'], CONFIG['sample_rate'], segment_len=None)

loader_kwargs = dict(
    num_workers=CONFIG['num_workers'],
    pin_memory=torch.cuda.is_available(),
    collate_fn=collate_fn,
)
if CONFIG['num_workers'] > 0:
    loader_kwargs.update(
        persistent_workers=CONFIG.get('persistent_workers', False),
        prefetch_factor=CONFIG.get('prefetch_factor', 2),
    )

train_loader = DataLoader(
    train_dataset, batch_size=CONFIG['batch_size'], shuffle=True,
    drop_last=True, **loader_kwargs
)
valid_loader = DataLoader(
    valid_dataset, batch_size=CONFIG.get('valid_batch_size', CONFIG['batch_size']), shuffle=False,
    drop_last=False, **loader_kwargs
)

print(f'Train: {len(train_dataset)} samples, {len(train_loader)} batches')
print(f'Valid: {len(valid_dataset)} samples, {len(valid_loader)} batches')
if aug_cfg:
    print(f'Augmentation: ON (gain={aug_cfg["aug_gain_range_db"]}, snr_remix={aug_cfg["aug_snr_remix_range"]}, prob={aug_cfg["aug_prob"]})')
else:
    print('Augmentation: OFF')


## 7. Khởi tạo Model, Optimizer, Scheduler

In [ ]:
import sys
import time
import json
from pathlib import Path
from torch.utils.tensorboard import SummaryWriter

# Đảm bảo import được deepvqe.py
work_dir = '/kaggle/working/DeepVQE_Workspace/deepvqe'
if work_dir not in sys.path:
    sys.path.insert(0, work_dir)

from deepvqe import DeepVQE

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU count: {torch.cuda.device_count()}')
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

# Tạo model
model = DeepVQE().to(device)

# Bật DataParallel nếu có nhiều GPU
if torch.cuda.device_count() > 1:
    print(f'Bật DataParallel trên {torch.cuda.device_count()} GPUs')
    model = torch.nn.DataParallel(model)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params: {total_params / 1e6:.2f}M | Trainable: {trainable_params / 1e6:.2f}M')

# Optimizer & Scheduler
optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG['lr'])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min',
    factor=CONFIG['scheduler_factor'],
    patience=CONFIG['scheduler_patience'],
    min_lr=CONFIG['scheduler_min_lr'],
)

# V2: Mixed Precision GradScaler
use_amp = CONFIG['use_amp'] and device.type == 'cuda'
scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
print(f'Mixed Precision (AMP): {"ON" if use_amp else "OFF"}')

# STFT window: sqrt-Hann, used consistently for STFT/ISTFT/inference.
window = torch.hann_window(CONFIG['win_length']).sqrt().to(device)

# Output dir
output_dir = Path(CONFIG['output_dir'])
output_dir.mkdir(parents=True, exist_ok=True)
with open(output_dir / 'config.json', 'w') as f:
    json.dump({k: str(v) if isinstance(v, tuple) else v for k, v in CONFIG.items()}, f, indent=2)

# TensorBoard writer is optional; writing to /kaggle/working is usually fine but still opt-in.
tb_log_dir = output_dir / 'tb_logs'
tb_writer = SummaryWriter(log_dir=str(tb_log_dir)) if CONFIG.get('tensorboard', False) else None
print(f'Output dir: {output_dir}')
print(f'TensorBoard: {"ON" if tb_writer else "OFF"}')
if tb_writer:
    print(f'TensorBoard log: {tb_log_dir}')


## 8. Hàm tiện ích: STFT, Loss, Checkpoint, Log, Metrics


In [ ]:
import csv

def make_stft(wav, n_fft, hop_length, win_length, win):
    """wav [B, T] -> spec [B, F, T_frames, 2]"""
    spec = torch.stft(wav, n_fft=n_fft, hop_length=hop_length, win_length=win_length,
                      window=win, return_complex=True)
    return torch.view_as_real(spec)

def make_istft(spec, n_fft, hop_length, win_length, win, length=None):
    """spec [B, F, T_frames, 2] -> wav [B, T]"""
    complex_spec = torch.complex(spec[..., 0], spec[..., 1])
    return torch.istft(complex_spec, n_fft=n_fft, hop_length=hop_length,
                       win_length=win_length, window=win, length=length)

def compute_loss(model, noisy_wav, clean_wav, cfg, win):
    """HybridLoss: Compressed RI + Compressed Mag + negative SI-SNR.
    V2: SI-SNR tính trên clean_wav gốc thay vì ISTFT(STFT(clean))."""
    n_fft = cfg['n_fft']
    hop = cfg['hop_length']
    win_len = cfg['win_length']
    c = cfg['compress_factor']
    
    noisy_spec = make_stft(noisy_wav, n_fft, hop, win_len, win)
    
    # Only autocast model forward; STFT/ISTFT/loss stay float32.
    amp_enabled = cfg.get('use_amp', False) and noisy_wav.device.type == 'cuda'
    with torch.amp.autocast('cuda', enabled=amp_enabled):
        pred_stft = model(noisy_spec)
    
    pred_stft = pred_stft.float()
    
    clean_spec = make_stft(clean_wav, n_fft, hop, win_len, win)
    min_t = min(pred_stft.shape[2], clean_spec.shape[2])
    pred_stft = pred_stft[:, :, :min_t, :]
    true_stft = clean_spec[:, :, :min_t, :]
    
    pred_stft_real, pred_stft_imag = pred_stft[:,:,:,0], pred_stft[:,:,:,1]
    true_stft_real, true_stft_imag = true_stft[:,:,:,0], true_stft[:,:,:,1]
    
    pred_mag = torch.sqrt(pred_stft_real**2 + pred_stft_imag**2 + 1e-12)
    true_mag = torch.sqrt(true_stft_real**2 + true_stft_imag**2 + 1e-12)
    
    pred_real_c = pred_stft_real / (pred_mag**(1 - c))
    pred_imag_c = pred_stft_imag / (pred_mag**(1 - c))
    true_real_c = true_stft_real / (true_mag**(1 - c))
    true_imag_c = true_stft_imag / (true_mag**(1 - c))
    
    real_loss = torch.mean((pred_real_c - true_real_c)**2)
    imag_loss = torch.mean((pred_imag_c - true_imag_c)**2)
    mag_loss = torch.mean((pred_mag**c - true_mag**c)**2)
    
    y_pred = torch.istft(pred_stft_real + 1j*pred_stft_imag, n_fft=n_fft, hop_length=hop, win_length=win_len, window=win)
    y_true = clean_wav
    
    min_wav_len = min(y_pred.shape[-1], y_true.shape[-1])
    y_pred = y_pred[..., :min_wav_len]
    y_true = y_true[..., :min_wav_len]
    
    y_target = torch.sum(y_true * y_pred, dim=-1, keepdim=True) * y_true / (torch.sum(torch.square(y_true), dim=-1, keepdim=True) + 1e-8)
    sisnr = -torch.log10(torch.sum(torch.square(y_target), dim=-1, keepdim=True) / torch.sum(torch.square(y_pred - y_target), dim=-1, keepdim=True) + 1e-8).mean()
    
    loss = cfg['lamda_ri'] * (real_loss + imag_loss) + cfg['lamda_mag'] * mag_loss + sisnr
    
    return loss, {'loss': loss.item(), 'ri_loss': (real_loss + imag_loss).item(), 'mag_loss': mag_loss.item(), 'sisnr': sisnr.item()}


def save_checkpoint(path, model, optimizer, scheduler, epoch, best_loss, bad_epochs, scaler=None):
    model_state = model.module.state_dict() if hasattr(model, 'module') else model.state_dict()
    ckpt = {
        'model': model_state,
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'epoch': epoch,
        'best_loss': best_loss,
        'bad_epochs': bad_epochs,
    }
    if scaler is not None:
        ckpt['scaler'] = scaler.state_dict()
    torch.save(ckpt, str(path))


def load_checkpoint(path, model, optimizer=None, scheduler=None, device='cpu', scaler=None):
    ckpt = torch.load(str(path), map_location=device, weights_only=False)
    target = model.module if hasattr(model, 'module') else model
    target.load_state_dict(ckpt['model'])
    if optimizer and 'optimizer' in ckpt:
        optimizer.load_state_dict(ckpt['optimizer'])
    if scheduler and ckpt.get('scheduler'):
        scheduler.load_state_dict(ckpt['scheduler'])
    if scaler and ckpt.get('scaler'):
        scaler.load_state_dict(ckpt['scaler'])
    return ckpt.get('epoch', 0), ckpt.get('best_loss'), ckpt.get('bad_epochs', 0)


def append_log(log_path, row_dict):
    log_path = Path(log_path)
    file_exists = log_path.exists() and log_path.stat().st_size > 0
    with open(log_path, 'a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=list(row_dict.keys()))
        if not file_exists:
            writer.writeheader()
        writer.writerow(row_dict)


def compute_pesq_stoi(model, dataloader, cfg, win, device, max_samples=200):
    from pesq import pesq
    from pystoi import stoi

    m = model.module if hasattr(model, 'module') else model
    m.eval()
    sr = cfg['sample_rate']
    pesq_scores, stoi_scores = [], []
    count = 0
    amp_enabled = cfg.get('use_amp', False) and device.type == 'cuda'

    with torch.no_grad():
        for batch in dataloader:
            noisy = batch['noisy'].to(device)
            clean = batch['clean']

            spec = make_stft(noisy, cfg['n_fft'], cfg['hop_length'], cfg['win_length'], win)
            with torch.amp.autocast('cuda', enabled=amp_enabled):
                pred = m(spec)
            enhanced = make_istft(pred.float(), cfg['n_fft'], cfg['hop_length'], cfg['win_length'], win)

            for i in range(enhanced.shape[0]):
                enh_np = enhanced[i].cpu().numpy()
                cln_np = clean[i].numpy()
                min_l = min(len(enh_np), len(cln_np))
                enh_np, cln_np = enh_np[:min_l], cln_np[:min_l]

                try:
                    pesq_scores.append(pesq(sr, cln_np, enh_np, 'wb'))
                except Exception:
                    pass
                stoi_scores.append(stoi(cln_np, enh_np, sr, extended=False))

                count += 1
                if count >= max_samples:
                    break
            if count >= max_samples:
                break

    return {
        'pesq': np.mean(pesq_scores) if pesq_scores else 0.0,
        'stoi': np.mean(stoi_scores) if stoi_scores else 0.0,
    }


print('Hàm tiện ích đã sẵn sàng.')


## 8.5 Khởi động TensorBoard (tùy chọn)
Chạy cell này để mở TensorBoard dashboard ngay trong Kaggle nếu `CONFIG['tensorboard'] = True`.


In [ ]:
# Mở TensorBoard dashboard nếu CONFIG['tensorboard'] = True.
if CONFIG.get('tensorboard', False):
    %load_ext tensorboard
    %tensorboard --logdir {str(output_dir / 'tb_logs')}
else:
    print('TensorBoard đang tắt trong CONFIG để ưu tiên tốc độ train.')


## 9. Vòng lặp huấn luyện (Training Loop)


In [ ]:
try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

# === Auto-resume nếu có checkpoint ===
start_epoch = 0
best_loss = None
bad_epochs = 0

resume_path = output_dir / 'last.pt'
if resume_path.exists():
    start_epoch, best_loss, bad_epochs = load_checkpoint(
        resume_path, model, optimizer, scheduler, device=device, scaler=scaler
    )
    print(f'Đã resume từ epoch {start_epoch}, best_loss={best_loss:.6f}, bad_epochs={bad_epochs}')
else:
    print('Bắt đầu training từ đầu.')

log_path = output_dir / 'train_log.csv'
print(f'Log file: {log_path}')
print(f'AMP: {"ON" if use_amp else "OFF"} | Batch: {CONFIG["batch_size"]} | Accum: {CONFIG["grad_accum_steps"]}')


# === TRAINING LOOP ===
accum_steps = CONFIG.get('grad_accum_steps', 1)
progress_update_every = max(1, CONFIG.get('progress_update_every', 10))

for epoch in range(start_epoch + 1, CONFIG['epochs'] + 1):
    # --- TRAIN ---
    model.train()
    train_losses = []
    t0 = time.time()
    optimizer.zero_grad(set_to_none=True)
    
    pbar = tqdm(train_loader, desc=f'Epoch {epoch} [Train]', leave=False)
    for batch_idx, batch in enumerate(pbar):
        noisy = batch['noisy'].to(device, non_blocking=True)
        clean = batch['clean'].to(device, non_blocking=True)
        
        loss, parts = compute_loss(model, noisy, clean, CONFIG, window)
        loss = loss / accum_steps
        
        scaler.scale(loss).backward()
        
        if (batch_idx + 1) % accum_steps == 0 or (batch_idx + 1) == len(train_loader):
            if CONFIG['grad_clip']:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['grad_clip'])
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
        
        train_losses.append(parts)
        if batch_idx % progress_update_every == 0 or (batch_idx + 1) == len(train_loader):
            pbar.set_postfix({'loss': f"{parts['loss']:.4f}"})
    
    avg_train = {k: np.mean([d[k] for d in train_losses]) for k in train_losses[0]}
    elapsed = time.time() - t0
    
    # --- VALID ---
    model.eval()
    valid_losses = []
    
    with torch.no_grad():
        for batch in tqdm(valid_loader, desc=f'Epoch {epoch} [Valid]', leave=False):
            noisy = batch['noisy'].to(device, non_blocking=True)
            clean = batch['clean'].to(device, non_blocking=True)
            _, parts = compute_loss(model, noisy, clean, CONFIG, window)
            valid_losses.append(parts)
    
    avg_valid = {k: np.mean([d[k] for d in valid_losses]) for k in valid_losses[0]}
    
    pesq_stoi = {'pesq': 0.0, 'stoi': 0.0}
    eval_every = CONFIG.get('eval_pesq_every', 0)
    if eval_every > 0 and epoch % eval_every == 0:
        print(f'  Đang tính PESQ/STOI (mỗi {eval_every} epoch)...')
        pesq_stoi = compute_pesq_stoi(model, valid_loader, CONFIG, window, device)
        print(f'  >> PESQ: {pesq_stoi["pesq"]:.3f} | STOI: {pesq_stoi["stoi"]:.4f}')
    
    # --- Scheduler ---
    scheduler.step(avg_valid['loss'])
    current_lr = optimizer.param_groups[0]['lr']
    
    # --- Logging ---
    print(
        f"Epoch {epoch:3d}/{CONFIG['epochs']} | "
        f"Train Loss: {avg_train['loss']:.6f} (ri={avg_train['ri_loss']:.4f}, mag={avg_train['mag_loss']:.4f}, sisnr={avg_train['sisnr']:.4f}) | "
        f"Valid Loss: {avg_valid['loss']:.6f} (ri={avg_valid['ri_loss']:.4f}, mag={avg_valid['mag_loss']:.4f}, sisnr={avg_valid['sisnr']:.4f}) | "
        f"LR: {current_lr:.2e} | Time: {elapsed:.0f}s"
    )
    
    append_log(log_path, {
        'epoch': epoch,
        'train_loss': f"{avg_train['loss']:.6f}",
        'train_ri_loss': f"{avg_train['ri_loss']:.6f}",
        'train_mag_loss': f"{avg_train['mag_loss']:.6f}",
        'train_sisnr': f"{avg_train['sisnr']:.6f}",
        'valid_loss': f"{avg_valid['loss']:.6f}",
        'valid_ri_loss': f"{avg_valid['ri_loss']:.6f}",
        'valid_mag_loss': f"{avg_valid['mag_loss']:.6f}",
        'valid_sisnr': f"{avg_valid['sisnr']:.6f}",
        'valid_pesq': f"{pesq_stoi['pesq']:.4f}",
        'valid_stoi': f"{pesq_stoi['stoi']:.4f}",
        'lr': f"{current_lr:.2e}",
        'time_s': f"{elapsed:.0f}",
    })
    
    if tb_writer:
        tb_writer.add_scalars('Loss/Total', {'train': avg_train['loss'], 'valid': avg_valid['loss']}, epoch)
        tb_writer.add_scalars('Loss/RI', {'train': avg_train['ri_loss'], 'valid': avg_valid['ri_loss']}, epoch)
        tb_writer.add_scalars('Loss/Mag', {'train': avg_train['mag_loss'], 'valid': avg_valid['mag_loss']}, epoch)
        tb_writer.add_scalars('Loss/SISNR', {'train': avg_train['sisnr'], 'valid': avg_valid['sisnr']}, epoch)
        tb_writer.add_scalar('LR', current_lr, epoch)
        if pesq_stoi['pesq'] > 0:
            tb_writer.add_scalar('Metrics/PESQ', pesq_stoi['pesq'], epoch)
            tb_writer.add_scalar('Metrics/STOI', pesq_stoi['stoi'], epoch)
        tb_writer.flush()
    
    # --- Checkpoint ---
    is_best = best_loss is None or avg_valid['loss'] < best_loss
    if is_best:
        best_loss = avg_valid['loss']
        bad_epochs = 0
    else:
        bad_epochs += 1
    
    save_checkpoint(output_dir / 'last.pt', model, optimizer, scheduler, epoch, best_loss, bad_epochs, scaler)
    
    if is_best:
        save_checkpoint(output_dir / 'best.pt', model, optimizer, scheduler, epoch, best_loss, bad_epochs, scaler)
        print(f'  >>> Saved best model (valid_loss={best_loss:.6f})')
    
    if bad_epochs >= CONFIG['early_stopping_patience']:
        print(f'Early stopping sau {bad_epochs} epoch không cải thiện. Best loss: {best_loss:.6f}')
        break

if tb_writer:
    tb_writer.close()
print(f'\n=== Huấn luyện hoàn tất! ===')
print(f'Best valid loss: {best_loss:.6f}')
print(f'Checkpoint: {output_dir}')
if tb_writer:
    print(f'TensorBoard log: {tb_log_dir}')


## 10. Kiểm tra nhanh sau training

In [ ]:
import IPython.display as ipd

# Load best model
best_ckpt = output_dir / 'best.pt'
if best_ckpt.exists():
    load_checkpoint(best_ckpt, model, device=device)
    print('Đã load best.pt')

model.eval()
test_df = pd.read_csv(CONFIG['test_csv'])
sample = test_df.iloc[0]

sig_noisy, sr = torchaudio.load(sample['noisy_wav'])
if sr != CONFIG['sample_rate']:
    sig_noisy = torchaudio.functional.resample(sig_noisy, sr, CONFIG['sample_rate'])

with torch.no_grad():
    noisy_wav = sig_noisy.squeeze(0).to(device)
    spec = make_stft(noisy_wav.unsqueeze(0), CONFIG['n_fft'], CONFIG['hop_length'], CONFIG['win_length'], window)
    m = model.module if hasattr(model, 'module') else model
    with torch.amp.autocast('cuda', enabled=use_amp):
        pred = m(spec)
    enhanced = make_istft(pred.float(), CONFIG['n_fft'], CONFIG['hop_length'], CONFIG['win_length'], window, length=noisy_wav.shape[0])

print(f'Sample: {sample["ID"]}')
print('Noisy:')
ipd.display(ipd.Audio(sig_noisy.squeeze().cpu().numpy(), rate=CONFIG['sample_rate']))
print('Enhanced:')
ipd.display(ipd.Audio(enhanced.squeeze().cpu().numpy(), rate=CONFIG['sample_rate']))

sig_clean, sr_c = torchaudio.load(sample['clean_wav'])
if sr_c != CONFIG['sample_rate']:
    sig_clean = torchaudio.functional.resample(sig_clean, sr_c, CONFIG['sample_rate'])
print('Clean (ground truth):')
ipd.display(ipd.Audio(sig_clean.squeeze().cpu().numpy(), rate=CONFIG['sample_rate']))


## 11. Tải checkpoint về máy

In [ ]:
from IPython.display import FileLink, display
from pathlib import Path
import shutil

checkpoint_dir = Path(CONFIG['output_dir'])
archive_base = Path('/kaggle/working/deepvqe_vctk_checkpoints')
archive_path = archive_base.with_suffix('.zip')

if not checkpoint_dir.exists():
    raise FileNotFoundError(f'Không tìm thấy thư mục checkpoint: {checkpoint_dir}')

if archive_path.exists():
    archive_path.unlink()

shutil.make_archive(str(archive_base), 'zip', root_dir=str(checkpoint_dir))

print(f'Đã nén checkpoint: {archive_path}')
print(f'Kích thước: {archive_path.stat().st_size / (1024 ** 2):.2f} MB')
display(FileLink(str(archive_path)))

## 12. Chạy ablation benchmarks
Cell này chạy smoke checks, benchmark kiến trúc cho các cấu hình trong `ablation/`, và có thể bật thêm train/eval/ONNX khi cần.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

import pandas as pd
import torch
from IPython.display import FileLink, display

# ===================== TÙY CHỈNH NHANH =====================
# Nếu muốn chạy ít cấu hình để smoke test trước, sửa thành: ['Baseline', 'B1a']
AB_CONFIGS = [
    'Baseline', 'B1a', 'B1b', 'B2', 'B3',
    'C1', 'C1a-g2', 'C1a-g4', 'C1b', 'C2', 'C3', 'C4',
]

RUN_SMOKE_TESTS = True
RUN_ARCH_BENCHMARK = True

# None = tự tìm model đã upload trong /kaggle/input/**/best.pt.
# Nếu Kaggle mount nhiều best.pt, điền path cụ thể vào đây.
BASELINE_CHECKPOINT = None

# Hai phần dưới tốn GPU lâu hơn. Bật True khi bạn muốn benchmark chất lượng sau train.
RUN_TRAINING = False
RUN_QUALITY_EVAL = True
RUN_ONNX_EXPORT = False

AB_EPOCHS = 1           # tăng lên khi chạy ablation nghiêm túc
AB_BATCH_SIZE = 8
AB_NUM_WORKERS = 0
AB_PIN_MEMORY = False
AB_EARLY_STOP_PATIENCE = 12
AB_EARLY_STOP_MIN_DELTA = 0.01
AB_EARLY_STOP_MIN_EPOCHS = 20
AB_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
RESUME_EXISTING = True

ARCH_FRAMES = 63
ARCH_WARMUP = 1
ARCH_REPEATS = 3
INSTALL_OPTIONAL_PACKAGES = False  # ptflops / pesq / pystoi / onnxruntime nếu cần
BENCHMARK_REPO = 'https://github.com/hoxuanphu/Pdeepvqe.git'
BENCHMARK_REPO_DIR = Path('/kaggle/working/Pdeepvqe_benchmark_src')
# ===========================================================

repo_dir = Path.cwd()
if not (repo_dir / 'deepvqe.py').exists():
    fallback_repo = Path('/kaggle/working/DeepVQE_Workspace/deepvqe')
    if (fallback_repo / 'deepvqe.py').exists():
        repo_dir = fallback_repo
        os.chdir(repo_dir)
    else:
        raise FileNotFoundError('Không tìm thấy deepvqe.py. Hãy chạy cell clone repo trước.')

print(f'Repo dir: {repo_dir}')
print(f'Device: {AB_DEVICE}')

def run_py(args, cwd=repo_dir):
    cmd = [sys.executable, *[str(arg) for arg in args]]
    print('\n$ ' + ' '.join(cmd), flush=True)
    result = subprocess.run(cmd, cwd=str(cwd), text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr, file=sys.stderr)
    if result.returncode != 0:
        raise RuntimeError('Lệnh lỗi với exit code ' + str(result.returncode) + ': ' + ' '.join(cmd))
    return result

def find_ablation_source():
    for root in [Path('/kaggle/input'), Path('/kaggle/working')]:
        if not root.exists():
            continue
        for marker in root.rglob('deepvqe_ablation.py'):
            if marker.parent.name == 'ablation':
                return marker.parent

    print(f'Không tìm thấy ablation local; clone benchmark repo: {BENCHMARK_REPO}')
    if not (BENCHMARK_REPO_DIR / 'ablation' / 'deepvqe_ablation.py').exists():
        if BENCHMARK_REPO_DIR.exists() and (BENCHMARK_REPO_DIR / '.git').exists():
            subprocess.run(['git', '-C', str(BENCHMARK_REPO_DIR), 'pull', '--ff-only'], check=False)
        elif not BENCHMARK_REPO_DIR.exists():
            subprocess.run(['git', 'clone', '--depth', '1', BENCHMARK_REPO, str(BENCHMARK_REPO_DIR)], check=True)

    source = BENCHMARK_REPO_DIR / 'ablation'
    if (source / 'deepvqe_ablation.py').exists():
        return source
    return None

if not (repo_dir / 'ablation' / 'deepvqe_ablation.py').exists():
    source_ablation = find_ablation_source()
    if source_ablation is None:
        raise FileNotFoundError(
            'Không tìm thấy ablation/deepvqe_ablation.py trong repo hiện tại. '
            f'Không clone được benchmark repo: {BENCHMARK_REPO}'
        )
    print(f'Copy ablation từ Kaggle input/working: {source_ablation}')
    shutil.copytree(source_ablation, repo_dir / 'ablation', dirs_exist_ok=True)
    source_root = source_ablation.parent
    if (source_root / 'stream').exists() and not (repo_dir / 'stream').exists():
        shutil.copytree(source_root / 'stream', repo_dir / 'stream', dirs_exist_ok=True)

if not (repo_dir / 'stream' / 'modules' / 'convolution.py').exists():
    raise FileNotFoundError('Thiếu stream/modules/convolution.py, ablation streaming benchmark sẽ không chạy được.')

def patch_eval_script_for_uploaded_checkpoints():
    path = repo_dir / 'ablation' / 'eval_ablation_quality.py'
    if not path.exists():
        return
    text = path.read_text(encoding='utf-8')
    original = text
    text = text.replace(
        'ckpt = torch.load(str(checkpoint_path), map_location=device)',
        "ckpt = torch.load(str(checkpoint_path), map_location='cpu', weights_only=False)",
    )
    text = text.replace(
        'model.load_state_dict(ckpt["model"])',
        "state = ckpt.get('model', ckpt.get('model_state_dict', ckpt.get('state_dict', ckpt)))\n"
        "    state = {k.replace('module.', '', 1): v for k, v in state.items()}\n"
        "    model.load_state_dict(state, strict=True)",
    )
    if text != original:
        path.write_text(text, encoding='utf-8')
        print(f'Đã patch {path} để load checkpoint upload an toàn hơn.')

patch_eval_script_for_uploaded_checkpoints()

results_dir = Path('/kaggle/working/DeepVQE_Workspace/results/ablation')
runs_dir = Path('/kaggle/working/DeepVQE_Workspace/runs/ablation')
onnx_dir = Path('/kaggle/working/DeepVQE_Workspace/onnx_models/ablation')
manifest_dir = Path('/kaggle/working/DeepVQE_Workspace/metadata/ablation_manifests')
for path in [results_dir, runs_dir, onnx_dir, manifest_dir]:
    path.mkdir(parents=True, exist_ok=True)

arch_csv = results_dir / 'ablation_arch_benchmark.csv'
quality_csv = results_dir / 'ablation_quality.csv'
onnx_csv = results_dir / 'ablation_onnx.csv'
summary_csv = results_dir / 'ablation_summary.csv'

def find_uploaded_best_checkpoint(explicit_path=None):
    if explicit_path:
        path = Path(explicit_path)
        if not path.exists():
            raise FileNotFoundError(f'Không tìm thấy BASELINE_CHECKPOINT: {path}')
        return path

    input_root = Path('/kaggle/input')
    if not input_root.exists():
        return None

    candidates = list(input_root.rglob('best.pt'))
    if not candidates:
        return None

    def score(path):
        lowered = ' '.join(part.lower() for part in path.parts)
        return (0 if 'deepvqe' in lowered else 1, len(str(path)), str(path))

    return sorted(candidates, key=score)[0]

uploaded_baseline_ckpt = find_uploaded_best_checkpoint(BASELINE_CHECKPOINT)
if uploaded_baseline_ckpt is not None:
    print(f'Dùng uploaded Baseline checkpoint: {uploaded_baseline_ckpt}')
else:
    print('Không tìm thấy uploaded best.pt trong /kaggle/input; Baseline eval/ONNX sẽ dùng checkpoint trong runs_dir nếu có.')

def checkpoint_for_config(config_id):
    if config_id == 'Baseline' and uploaded_baseline_ckpt is not None:
        return uploaded_baseline_ckpt
    return runs_dir / config_id / 'best.pt'

if INSTALL_OPTIONAL_PACKAGES:
    packages = ['ptflops']
    if RUN_QUALITY_EVAL:
        packages += ['pesq', 'pystoi']
    if RUN_ONNX_EXPORT:
        packages += ['onnx', 'onnxruntime', 'onnxsim']
    run_py(['-m', 'pip', 'install', '-q', *packages])

def csv_to_jsonl(csv_path, jsonl_path):
    df = pd.read_csv(csv_path)
    required = {'ID', 'clean_wav', 'noisy_wav'}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f'{csv_path} thiếu cột: {sorted(missing)}')
    with open(jsonl_path, 'w', encoding='utf-8') as f:
        for row in df.to_dict('records'):
            item = {
                'id': row['ID'],
                'mixture': row['noisy_wav'],
                'target': row['clean_wav'],
                'noisy_wav': row['noisy_wav'],
                'clean_wav': row['clean_wav'],
            }
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    return jsonl_path

metadata_base = Path(globals().get('metadata_dir', '/kaggle/working/DeepVQE_Workspace/metadata'))
csv_manifests = {
    'train': metadata_base / 'train.csv',
    'valid': metadata_base / 'valid.csv',
    'test': metadata_base / 'test.csv',
}
jsonl_manifests = {}
if all(path.exists() for path in csv_manifests.values()):
    for split, csv_path in csv_manifests.items():
        jsonl_manifests[split] = csv_to_jsonl(csv_path, manifest_dir / f'{split}.jsonl')
    print('Đã tạo manifest JSONL cho ablation:')
    for split, path in jsonl_manifests.items():
        print(f'  {split}: {path}')
elif RUN_TRAINING or RUN_QUALITY_EVAL:
    raise FileNotFoundError('Thiếu train/valid/test CSV. Hãy chạy cell tạo metadata trước.')
else:
    print('Chưa có metadata CSV; bỏ qua manifest train/eval và chỉ chạy benchmark kiến trúc.')

if RUN_SMOKE_TESTS:
    run_py(['ablation/test_ablation_baseline.py'])
    run_py(['ablation/test_ablation_streaming.py', '--configs', *AB_CONFIGS])

if RUN_ARCH_BENCHMARK:
    run_py([
        'ablation/run_ablation_benchmark.py',
        '--output', arch_csv,
        '--configs', *AB_CONFIGS,
        '--device', AB_DEVICE,
        '--frames', ARCH_FRAMES,
        '--warmup', ARCH_WARMUP,
        '--repeats', ARCH_REPEATS,
    ])

if RUN_TRAINING:
    for config_id in AB_CONFIGS:
        output_dir = runs_dir / config_id
        args = [
            'ablation/train_ablation.py',
            '--config-id', config_id,
            '--train-manifest', jsonl_manifests['train'],
            '--valid-manifest', jsonl_manifests['valid'],
            '--output-dir', output_dir,
            '--device', AB_DEVICE,
            '--epochs', AB_EPOCHS,
            '--batch-size', AB_BATCH_SIZE,
            '--num-workers', AB_NUM_WORKERS,
            '--early-stop-patience', AB_EARLY_STOP_PATIENCE,
            '--early-stop-min-delta', AB_EARLY_STOP_MIN_DELTA,
            '--early-stop-min-epochs', AB_EARLY_STOP_MIN_EPOCHS,
        ]
        last_ckpt = output_dir / 'last.pt'
        if RESUME_EXISTING and last_ckpt.exists():
            args += ['--resume', last_ckpt]
        run_py(args)

if RUN_QUALITY_EVAL:
    for config_id in AB_CONFIGS:
        ckpt = checkpoint_for_config(config_id)
        if not ckpt.exists():
            print(f'Bỏ qua eval {config_id}: chưa có checkpoint {ckpt}')
            continue
        run_py([
            'ablation/eval_ablation_quality.py',
            '--config-id', config_id,
            '--checkpoint', ckpt,
            '--manifest', jsonl_manifests['test'],
            '--output', quality_csv,
            '--device', AB_DEVICE,
        ])

if RUN_ONNX_EXPORT:
    for config_id in AB_CONFIGS:
        ckpt = checkpoint_for_config(config_id)
        if not ckpt.exists():
            print(f'Bỏ qua ONNX {config_id}: chưa có checkpoint {ckpt}')
            continue
        run_py([
            'ablation/export_ablation_onnx.py',
            '--config-id', config_id,
            '--checkpoint', ckpt,
            '--output-dir', onnx_dir,
            '--results', onnx_csv,
            '--device', 'cpu',
        ])

run_py([
    'ablation/collect_ablation_results.py',
    '--arch', arch_csv,
    '--quality', quality_csv,
    '--onnx', onnx_csv,
    '--output', summary_csv,
])

print('\nCSV kết quả:')
for csv_path in [arch_csv, quality_csv, onnx_csv, summary_csv]:
    if csv_path.exists():
        print(f'  {csv_path} ({csv_path.stat().st_size / 1024:.1f} KB)')
        display(pd.read_csv(csv_path).head(30))

archive_base = Path('/kaggle/working/ablation_benchmark_results')
archive_path = archive_base.with_suffix('.zip')
if archive_path.exists():
    archive_path.unlink()
shutil.make_archive(str(archive_base), 'zip', root_dir=str(results_dir))
print(f'Đã nén kết quả benchmark: {archive_path}')
display(FileLink(str(archive_path)))

## 13. Quality benchmark với `eval_ablation_quality.py`
Cell này chỉ chạy benchmark chất lượng cho checkpoint đã upload, dùng cùng script `ablation/eval_ablation_quality.py`.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

import pandas as pd
import torch
from IPython.display import FileLink, display

# ===================== TÙY CHỈNH NHANH =====================
EVAL_CONFIG_ID = 'Baseline'
EVAL_CHECKPOINT = None  # None = tự tìm /kaggle/input/**/best.pt từ model upload
EVAL_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
EVAL_MAX_TEST_ITEMS = None  # ví dụ 50 để smoke test nhanh; None = full test set
INSTALL_PESQ_STOI = False  # True nếu muốn thêm PESQ/STOI ngoài SI-SDR
BENCHMARK_REPO = 'https://github.com/hoxuanphu/Pdeepvqe.git'
BENCHMARK_REPO_DIR = Path('/kaggle/working/Pdeepvqe_benchmark_src')

EVAL_OUTPUT_CSV = Path('/kaggle/working/DeepVQE_Workspace/results/ablation/ablation_quality_uploaded_model.csv')
EVAL_MANIFEST_JSONL = Path('/kaggle/working/DeepVQE_Workspace/metadata/ablation_manifests/test_eval_ablation_quality.jsonl')
# ===========================================================

repo_dir = Path.cwd()
if not (repo_dir / 'deepvqe.py').exists():
    fallback_repo = Path('/kaggle/working/DeepVQE_Workspace/deepvqe')
    if (fallback_repo / 'deepvqe.py').exists():
        repo_dir = fallback_repo
        os.chdir(repo_dir)
    else:
        raise FileNotFoundError('Không tìm thấy deepvqe.py. Hãy chạy cell clone repo trước.')

def run_py(args, cwd=repo_dir):
    cmd = [sys.executable, *[str(arg) for arg in args]]
    print('\n$ ' + ' '.join(cmd), flush=True)
    result = subprocess.run(cmd, cwd=str(cwd), text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr, file=sys.stderr)
    if result.returncode != 0:
        raise RuntimeError('Lệnh lỗi với exit code ' + str(result.returncode) + ': ' + ' '.join(cmd))
    return result

def find_ablation_source():
    for root in [Path('/kaggle/input'), Path('/kaggle/working')]:
        if not root.exists():
            continue
        for marker in root.rglob('deepvqe_ablation.py'):
            if marker.parent.name == 'ablation':
                return marker.parent

    print(f'Không tìm thấy ablation local; clone benchmark repo: {BENCHMARK_REPO}')
    if not (BENCHMARK_REPO_DIR / 'ablation' / 'deepvqe_ablation.py').exists():
        if BENCHMARK_REPO_DIR.exists() and (BENCHMARK_REPO_DIR / '.git').exists():
            subprocess.run(['git', '-C', str(BENCHMARK_REPO_DIR), 'pull', '--ff-only'], check=False)
        elif not BENCHMARK_REPO_DIR.exists():
            subprocess.run(['git', 'clone', '--depth', '1', BENCHMARK_REPO, str(BENCHMARK_REPO_DIR)], check=True)

    source = BENCHMARK_REPO_DIR / 'ablation'
    if (source / 'deepvqe_ablation.py').exists():
        return source
    return None

if not (repo_dir / 'ablation' / 'eval_ablation_quality.py').exists():
    source_ablation = find_ablation_source()
    if source_ablation is None:
        raise FileNotFoundError(
            'Không tìm thấy ablation/eval_ablation_quality.py. '
            f'Không clone được benchmark repo: {BENCHMARK_REPO}'
        )
    print(f'Copy ablation từ: {source_ablation}')
    shutil.copytree(source_ablation, repo_dir / 'ablation', dirs_exist_ok=True)
    source_root = source_ablation.parent
    if (source_root / 'stream').exists() and not (repo_dir / 'stream').exists():
        shutil.copytree(source_root / 'stream', repo_dir / 'stream', dirs_exist_ok=True)

if not (repo_dir / 'stream' / 'modules' / 'convolution.py').exists():
    raise FileNotFoundError('Thiếu stream/modules/convolution.py, cần cho DeepVQE_Ablation import.')

def patch_eval_script_for_uploaded_checkpoints():
    path = repo_dir / 'ablation' / 'eval_ablation_quality.py'
    if not path.exists():
        return
    text = path.read_text(encoding='utf-8')
    original = text
    text = text.replace(
        'ckpt = torch.load(str(checkpoint_path), map_location=device)',
        "ckpt = torch.load(str(checkpoint_path), map_location='cpu', weights_only=False)",
    )
    text = text.replace(
        'model.load_state_dict(ckpt["model"])',
        "state = ckpt.get('model', ckpt.get('model_state_dict', ckpt.get('state_dict', ckpt)))\n"
        "    state = {k.replace('module.', '', 1): v for k, v in state.items()}\n"
        "    model.load_state_dict(state, strict=True)",
    )
    if text != original:
        path.write_text(text, encoding='utf-8')
        print(f'Đã patch {path} để load checkpoint upload an toàn hơn.')

patch_eval_script_for_uploaded_checkpoints()

def find_uploaded_checkpoint(explicit_path=None):
    if explicit_path:
        path = Path(explicit_path)
        if not path.exists():
            raise FileNotFoundError(f'Không tìm thấy EVAL_CHECKPOINT: {path}')
        return path

    input_root = Path('/kaggle/input')
    candidates = list(input_root.rglob('best.pt')) if input_root.exists() else []
    if not candidates:
        working_root = Path('/kaggle/working/DeepVQE_Workspace')
        candidates = list(working_root.rglob('best.pt')) if working_root.exists() else []
    if not candidates:
        raise FileNotFoundError('Không tìm thấy best.pt trong /kaggle/input hoặc DeepVQE_Workspace.')

    def score(path):
        lowered = ' '.join(part.lower() for part in path.parts)
        return (0 if 'deepvqe' in lowered else 1, len(str(path)), str(path))

    return sorted(candidates, key=score)[0]

def test_csv_to_jsonl(csv_path, jsonl_path, max_items=None):
    df = pd.read_csv(csv_path)
    required = {'ID', 'clean_wav', 'noisy_wav'}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f'{csv_path} thiếu cột: {sorted(missing)}')
    if max_items is not None:
        df = df.head(int(max_items))

    jsonl_path.parent.mkdir(parents=True, exist_ok=True)
    with jsonl_path.open('w', encoding='utf-8') as f:
        for row in df.to_dict('records'):
            item = {
                'id': row['ID'],
                'mixture': row['noisy_wav'],
                'target': row['clean_wav'],
                'noisy_wav': row['noisy_wav'],
                'clean_wav': row['clean_wav'],
            }
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    return jsonl_path, len(df)

if INSTALL_PESQ_STOI:
    run_py(['-m', 'pip', 'install', '-q', 'pesq', 'pystoi'])

metadata_base = Path(globals().get('metadata_dir', '/kaggle/working/DeepVQE_Workspace/metadata'))
test_csv = metadata_base / 'test.csv'
if not test_csv.exists():
    raise FileNotFoundError('Không tìm thấy test.csv. Hãy chạy cell tạo metadata trước.')

checkpoint_path = find_uploaded_checkpoint(EVAL_CHECKPOINT)
manifest_path, manifest_count = test_csv_to_jsonl(test_csv, EVAL_MANIFEST_JSONL, EVAL_MAX_TEST_ITEMS)
EVAL_OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

print(f'Repo dir: {repo_dir}')
print(f'Checkpoint: {checkpoint_path}')
print(f'Manifest: {manifest_path} ({manifest_count} mẫu)')
print(f'Output CSV: {EVAL_OUTPUT_CSV}')
print(f'Device: {EVAL_DEVICE}')

run_py([
    'ablation/eval_ablation_quality.py',
    '--config-id', EVAL_CONFIG_ID,
    '--checkpoint', checkpoint_path,
    '--manifest', manifest_path,
    '--output', EVAL_OUTPUT_CSV,
    '--device', EVAL_DEVICE,
])

df_quality = pd.read_csv(EVAL_OUTPUT_CSV)
display(df_quality)
display(FileLink(str(EVAL_OUTPUT_CSV)))

## 14. Phase 1 Training: B1a, B1b, C1

Cell này train vòng 1 theo `ablation/docs/phase1_training_plan.md` và lưu artifact theo run tag không có ngày tháng.

In [ ]:
from pathlib import Path
import subprocess
import os

repo_dir = Path('/kaggle/working/DeepVQE_Workspace/deepvqe')
if repo_dir.exists():
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only'], check=True)
    os.chdir(repo_dir)
else:
    subprocess.run([
        'git', 'clone',
        'https://github.com/hoxuanphu/Pdeepvqe.git',
        str(repo_dir)
    ], check=True)
    os.chdir(repo_dir)

print('Repo:', os.getcwd())

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

import pandas as pd
import torch
from IPython.display import FileLink, display

# ===================== PHASE 1 TRAINING =====================
PHASE1_RUN_TAG = 'phase1_b1a_b1b_c1_vctk_demand_e80_bs8'
PHASE1_TRAIN_CONFIGS = ['B1a', 'B1b', 'C1']
PHASE1_ARCH_CONFIGS = ['Baseline', *PHASE1_TRAIN_CONFIGS]

PHASE1_RUN_SMOKE_TESTS = True
PHASE1_RUN_ARCH_BENCHMARK = True
PHASE1_RUN_TRAINING = True
PHASE1_RUN_QUALITY_EVAL = True
PHASE1_RUN_ONNX_EXPORT = False

PHASE1_EPOCHS = 80
PHASE1_BATCH_SIZE = 8
PHASE1_NUM_WORKERS = 0
PHASE1_PIN_MEMORY = False
PHASE1_EARLY_STOP_PATIENCE = 12
PHASE1_EARLY_STOP_MIN_DELTA = 0.01
PHASE1_EARLY_STOP_MIN_EPOCHS = 20
PHASE1_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
PHASE1_USE_DATA_PARALLEL = torch.cuda.is_available() and torch.cuda.device_count() > 1
PHASE1_RESUME_EXISTING = True
PHASE1_INSTALL_OPTIONAL_PACKAGES = True
PHASE1_EVAL_BASELINE_IF_FOUND = True
PHASE1_BASELINE_CHECKPOINT = None  # Điền path nếu muốn eval lại Baseline upload.
PHASE1_EVAL_MAX_TEST_ITEMS = None  # Ví dụ 50 để smoke test quality nhanh.

PHASE1_ARCH_FRAMES = 63
PHASE1_ARCH_WARMUP = 1
PHASE1_ARCH_REPEATS = 3
BENCHMARK_REPO = 'https://github.com/hoxuanphu/Pdeepvqe.git'
BENCHMARK_REPO_DIR = Path('/kaggle/working/Pdeepvqe_benchmark_src')
# ============================================================

workspace_dir = Path('/kaggle/working/DeepVQE_Workspace')
repo_dir = Path.cwd()
if not (repo_dir / 'deepvqe.py').exists():
    fallback_repo = workspace_dir / 'deepvqe'
    if (fallback_repo / 'deepvqe.py').exists():
        repo_dir = fallback_repo
        os.chdir(repo_dir)
    else:
        raise FileNotFoundError('Không tìm thấy deepvqe.py. Hãy chạy cell clone repo trước.')

print(f'Repo dir: {repo_dir}')
print(f'Run tag: {PHASE1_RUN_TAG}')
print(f'Device: {PHASE1_DEVICE}')
if torch.cuda.is_available():
    print(f'CUDA GPUs: {torch.cuda.device_count()}')
    for gpu_idx in range(torch.cuda.device_count()):
        print(f'  GPU {gpu_idx}: {torch.cuda.get_device_name(gpu_idx)}')
print(f'DataParallel: {PHASE1_USE_DATA_PARALLEL}')

def run_py(args, cwd=repo_dir):
    cmd = [sys.executable, *[str(arg) for arg in args]]
    print('\n$ ' + ' '.join(cmd), flush=True)
    result = subprocess.run(cmd, cwd=str(cwd))
    if result.returncode != 0:
        raise RuntimeError('Lệnh lỗi với exit code ' + str(result.returncode) + ': ' + ' '.join(cmd))
    return result

def find_ablation_source():
    for root in [Path('/kaggle/input'), Path('/kaggle/working')]:
        if not root.exists():
            continue
        for marker in root.rglob('deepvqe_ablation.py'):
            if marker.parent.name == 'ablation':
                return marker.parent

    print(f'Không tìm thấy ablation local; clone benchmark repo: {BENCHMARK_REPO}')
    if not (BENCHMARK_REPO_DIR / 'ablation' / 'deepvqe_ablation.py').exists():
        if (BENCHMARK_REPO_DIR / '.git').exists():
            subprocess.run(['git', '-C', str(BENCHMARK_REPO_DIR), 'pull', '--ff-only'], check=False)
        elif not BENCHMARK_REPO_DIR.exists():
            subprocess.run(['git', 'clone', '--depth', '1', BENCHMARK_REPO, str(BENCHMARK_REPO_DIR)], check=True)

    source = BENCHMARK_REPO_DIR / 'ablation'
    if (source / 'deepvqe_ablation.py').exists():
        return source
    return None

if not (repo_dir / 'ablation' / 'deepvqe_ablation.py').exists():
    source_ablation = find_ablation_source()
    if source_ablation is None:
        raise FileNotFoundError(
            'Không tìm thấy ablation/deepvqe_ablation.py trong repo hiện tại. '
            f'Không clone được benchmark repo: {BENCHMARK_REPO}'
        )
    print(f'Copy ablation từ Kaggle input/working: {source_ablation}')
    shutil.copytree(source_ablation, repo_dir / 'ablation', dirs_exist_ok=True)
    source_root = source_ablation.parent
    if (source_root / 'stream').exists() and not (repo_dir / 'stream').exists():
        shutil.copytree(source_root / 'stream', repo_dir / 'stream', dirs_exist_ok=True)

if not (repo_dir / 'stream' / 'modules' / 'convolution.py').exists():
    raise FileNotFoundError('Thiếu stream/modules/convolution.py, ablation streaming benchmark sẽ không chạy được.')

if PHASE1_INSTALL_OPTIONAL_PACKAGES:
    packages = []
    if PHASE1_RUN_ARCH_BENCHMARK:
        packages.append('ptflops')
    if PHASE1_RUN_QUALITY_EVAL:
        packages += ['pesq', 'pystoi']
    if PHASE1_RUN_ONNX_EXPORT:
        packages += ['onnx', 'onnxruntime', 'onnxsim']
    packages = list(dict.fromkeys(packages))
    if packages:
        try:
            run_py(['-m', 'pip', 'install', '-q', *packages])
        except RuntimeError as exc:
            print(f'Cảnh báo: cài optional packages lỗi: {exc}')
            print('Vẫn tiếp tục; metric phụ thuộc package thiếu có thể bị bỏ trống.')

phase1_results_dir = workspace_dir / 'results' / 'ablation' / PHASE1_RUN_TAG
phase1_runs_dir = workspace_dir / 'runs' / 'ablation' / PHASE1_RUN_TAG
phase1_onnx_dir = workspace_dir / 'onnx_models' / 'ablation' / PHASE1_RUN_TAG
phase1_manifest_dir = workspace_dir / 'metadata' / 'ablation_manifests' / PHASE1_RUN_TAG
for path in [phase1_results_dir, phase1_runs_dir, phase1_onnx_dir, phase1_manifest_dir]:
    path.mkdir(parents=True, exist_ok=True)

phase1_arch_csv = phase1_results_dir / 'ablation_arch_benchmark.csv'
phase1_quality_csv = phase1_results_dir / 'ablation_quality.csv'
phase1_onnx_csv = phase1_results_dir / 'ablation_onnx.csv'
phase1_summary_csv = phase1_results_dir / 'ablation_summary.csv'

def csv_to_jsonl(csv_path, jsonl_path, max_items=None):
    df = pd.read_csv(csv_path)
    required = {'ID', 'clean_wav', 'noisy_wav'}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f'{csv_path} thiếu cột: {sorted(missing)}')
    if max_items is not None:
        df = df.head(int(max_items))

    jsonl_path.parent.mkdir(parents=True, exist_ok=True)
    with jsonl_path.open('w', encoding='utf-8') as f:
        for row in df.to_dict('records'):
            item = {
                'id': row['ID'],
                'mixture': row['noisy_wav'],
                'target': row['clean_wav'],
                'noisy_wav': row['noisy_wav'],
                'clean_wav': row['clean_wav'],
            }
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    return jsonl_path, len(df)

metadata_base = Path(globals().get('metadata_dir', workspace_dir / 'metadata'))
csv_manifests = {
    'train': metadata_base / 'train.csv',
    'valid': metadata_base / 'valid.csv',
    'test': metadata_base / 'test.csv',
}
jsonl_manifests = {}
if all(path.exists() for path in csv_manifests.values()):
    jsonl_manifests['train'], train_count = csv_to_jsonl(csv_manifests['train'], phase1_manifest_dir / 'train.jsonl')
    jsonl_manifests['valid'], valid_count = csv_to_jsonl(csv_manifests['valid'], phase1_manifest_dir / 'valid.jsonl')
    jsonl_manifests['test'], test_count = csv_to_jsonl(
        csv_manifests['test'],
        phase1_manifest_dir / 'test.jsonl',
        max_items=PHASE1_EVAL_MAX_TEST_ITEMS,
    )
    print('Manifest Phase 1:')
    print(f'  train: {jsonl_manifests["train"]} ({train_count} mẫu)')
    print(f'  valid: {jsonl_manifests["valid"]} ({valid_count} mẫu)')
    print(f'  test : {jsonl_manifests["test"]} ({test_count} mẫu)')
elif PHASE1_RUN_TRAINING or PHASE1_RUN_QUALITY_EVAL:
    raise FileNotFoundError('Thiếu train/valid/test CSV. Hãy chạy cell tạo metadata trước.')

def find_uploaded_best_checkpoint(explicit_path=None):
    if explicit_path:
        path = Path(explicit_path)
        if not path.exists():
            raise FileNotFoundError(f'Không tìm thấy PHASE1_BASELINE_CHECKPOINT: {path}')
        return path

    input_root = Path('/kaggle/input')
    candidates = list(input_root.rglob('best.pt')) if input_root.exists() else []
    if not candidates:
        return None

    def score(path):
        lowered = ' '.join(part.lower() for part in path.parts)
        return (0 if 'deepvqe' in lowered else 1, len(str(path)), str(path))

    return sorted(candidates, key=score)[0]

phase1_baseline_ckpt = find_uploaded_best_checkpoint(PHASE1_BASELINE_CHECKPOINT)
if phase1_baseline_ckpt is not None:
    print(f'Baseline checkpoint để eval so sánh: {phase1_baseline_ckpt}')
else:
    print('Không tìm thấy Baseline best.pt trong /kaggle/input; Phase 1 chỉ eval các checkpoint vừa train.')

def phase1_checkpoint_for_config(config_id):
    if config_id == 'Baseline' and phase1_baseline_ckpt is not None:
        return phase1_baseline_ckpt
    return phase1_runs_dir / config_id / 'best.pt'

if PHASE1_RUN_SMOKE_TESTS:
    run_py(['ablation/test_ablation_baseline.py'])
    run_py(['ablation/test_ablation_streaming.py', '--configs', *PHASE1_ARCH_CONFIGS])

if PHASE1_RUN_ARCH_BENCHMARK:
    run_py([
        'ablation/run_ablation_benchmark.py',
        '--output', phase1_arch_csv,
        '--configs', *PHASE1_ARCH_CONFIGS,
        '--device', PHASE1_DEVICE,
        '--frames', PHASE1_ARCH_FRAMES,
        '--warmup', PHASE1_ARCH_WARMUP,
        '--repeats', PHASE1_ARCH_REPEATS,
    ])

if PHASE1_RUN_TRAINING:
    for config_id in PHASE1_TRAIN_CONFIGS:
        output_dir = phase1_runs_dir / config_id
        args = [
            'ablation/train_ablation.py',
            '--config-id', config_id,
            '--train-manifest', jsonl_manifests['train'],
            '--valid-manifest', jsonl_manifests['valid'],
            '--output-dir', output_dir,
            '--device', PHASE1_DEVICE,
            '--epochs', PHASE1_EPOCHS,
            '--batch-size', PHASE1_BATCH_SIZE,
            '--num-workers', PHASE1_NUM_WORKERS,
            '--early-stop-patience', PHASE1_EARLY_STOP_PATIENCE,
            '--early-stop-min-delta', PHASE1_EARLY_STOP_MIN_DELTA,
            '--early-stop-min-epochs', PHASE1_EARLY_STOP_MIN_EPOCHS,
        ]
        last_ckpt = output_dir / 'last.pt'
        if not PHASE1_PIN_MEMORY:
            args += ['--no-pin-memory']
        if PHASE1_USE_DATA_PARALLEL:
            args += ['--data-parallel']
        if PHASE1_RESUME_EXISTING and last_ckpt.exists():
            args += ['--resume', last_ckpt, '--ignore-bad-resume']
        run_py(args)

if PHASE1_RUN_QUALITY_EVAL:
    eval_configs = list(PHASE1_TRAIN_CONFIGS)
    if PHASE1_EVAL_BASELINE_IF_FOUND and phase1_baseline_ckpt is not None:
        eval_configs = ['Baseline', *eval_configs]

    for config_id in eval_configs:
        ckpt = phase1_checkpoint_for_config(config_id)
        if not ckpt.exists():
            print(f'Bỏ qua eval {config_id}: chưa có checkpoint {ckpt}')
            continue
        run_py([
            'ablation/eval_ablation_quality.py',
            '--config-id', config_id,
            '--checkpoint', ckpt,
            '--manifest', jsonl_manifests['test'],
            '--output', phase1_quality_csv,
            '--device', PHASE1_DEVICE,
        ])

if PHASE1_RUN_ONNX_EXPORT:
    for config_id in PHASE1_TRAIN_CONFIGS:
        ckpt = phase1_checkpoint_for_config(config_id)
        if not ckpt.exists():
            print(f'Bỏ qua ONNX {config_id}: chưa có checkpoint {ckpt}')
            continue
        run_py([
            'ablation/export_ablation_onnx.py',
            '--config-id', config_id,
            '--checkpoint', ckpt,
            '--output-dir', phase1_onnx_dir,
            '--results', phase1_onnx_csv,
            '--device', 'cpu',
        ])

run_py([
    'ablation/collect_ablation_results.py',
    '--arch', phase1_arch_csv,
    '--quality', phase1_quality_csv,
    '--onnx', phase1_onnx_csv,
    '--output', phase1_summary_csv,
])

print('\nCSV Phase 1:')
for csv_path in [phase1_arch_csv, phase1_quality_csv, phase1_onnx_csv, phase1_summary_csv]:
    if csv_path.exists():
        print(f'  {csv_path} ({csv_path.stat().st_size / 1024:.1f} KB)')
        display(pd.read_csv(csv_path).head(30))

print('\nCheckpoint Phase 1:')
for config_id in PHASE1_TRAIN_CONFIGS:
    output_dir = phase1_runs_dir / config_id
    print(f'  {config_id}: best={output_dir / "best.pt"} | last={output_dir / "last.pt"}')

archive_base = Path('/kaggle/working') / f'{PHASE1_RUN_TAG}_results'
archive_path = archive_base.with_suffix('.zip')
if archive_path.exists():
    archive_path.unlink()
shutil.make_archive(str(archive_base), 'zip', root_dir=str(phase1_results_dir))
print(f'Đã nén CSV Phase 1: {archive_path}')
display(FileLink(str(archive_path)))

## 15. Tải checkpoint Phase 1 về máy

Cell này nén `best.pt`, `last.pt` và `config.json` của các config Phase 1 để tải về máy.

In [ ]:
from pathlib import Path
import shutil

from IPython.display import FileLink, display

PHASE1_RUN_TAG = globals().get('PHASE1_RUN_TAG', 'phase1_b1a_b1b_c1_vctk_demand_e80_bs8')
PHASE1_TRAIN_CONFIGS = globals().get('PHASE1_TRAIN_CONFIGS', ['B1a', 'B1b', 'C1'])
workspace_dir = globals().get('workspace_dir', None)
if workspace_dir is None:
    workspace_dir = globals().get('WORKSPACE', Path('/kaggle/working/DeepVQE_Workspace'))
workspace_dir = Path(workspace_dir)

phase1_runs_dir = workspace_dir / 'runs' / 'ablation' / PHASE1_RUN_TAG
if not phase1_runs_dir.exists():
    raise FileNotFoundError(f'Không tìm thấy thư mục checkpoint Phase 1: {phase1_runs_dir}')

missing = []
for config_id in PHASE1_TRAIN_CONFIGS:
    for filename in ['best.pt', 'last.pt', 'config.json']:
        path = phase1_runs_dir / config_id / filename
        if not path.exists():
            missing.append(str(path))

if missing:
    print('Cảnh báo: thiếu một số file checkpoint/artifact:')
    for item in missing:
        print('  ' + item)

archive_base = Path('/kaggle/working') / f'{PHASE1_RUN_TAG}_checkpoints'
archive_path = archive_base.with_suffix('.zip')
if archive_path.exists():
    archive_path.unlink()

shutil.make_archive(str(archive_base), 'zip', root_dir=str(phase1_runs_dir))
print(f'Đã nén checkpoint Phase 1: {archive_path}')
print(f'Nguồn: {phase1_runs_dir}')
display(FileLink(str(archive_path)))

## 16. Khôi phục checkpoint Phase 1 từ zip

Cell này giải nén zip checkpoint Phase 1 đã tải/lưu trước đó và ghi đè ngược lại vào thư mục run hiện tại. Trước khi ghi đè, nó tự backup thư mục hiện tại.

In [ ]:
from pathlib import Path
import shutil
import zipfile

PHASE1_RUN_TAG = globals().get('PHASE1_RUN_TAG', 'phase1_b1a_b1b_c1_vctk_demand_e80_bs8')
PHASE1_TRAIN_CONFIGS = globals().get('PHASE1_TRAIN_CONFIGS', ['B1a', 'B1b', 'C1'])
workspace_dir = Path(globals().get('workspace_dir', '/kaggle/working/DeepVQE_Workspace'))
phase1_runs_dir = workspace_dir / 'runs' / 'ablation' / PHASE1_RUN_TAG

# Sửa path này nếu zip nằm ở nơi khác, ví dụ /kaggle/input/.../phase1..._checkpoints.zip
RESTORE_ZIP_PATH = Path('/kaggle/working/phase1_b1a_b1b_c1_vctk_demand_e80_bs8_checkpoints.zip')
RESTORE_OVERWRITE = True

if not RESTORE_ZIP_PATH.exists():
    raise FileNotFoundError(f'Không tìm thấy zip restore: {RESTORE_ZIP_PATH}')
if not zipfile.is_zipfile(RESTORE_ZIP_PATH):
    raise zipfile.BadZipFile(f'File không phải zip hợp lệ: {RESTORE_ZIP_PATH}')

backup_dir = phase1_runs_dir.with_name(phase1_runs_dir.name + '_backup_before_restore')
if phase1_runs_dir.exists():
    if backup_dir.exists():
        shutil.rmtree(backup_dir)
    shutil.copytree(phase1_runs_dir, backup_dir)
    print(f'Đã backup checkpoint hiện tại sang: {backup_dir}')

phase1_runs_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(RESTORE_ZIP_PATH, 'r') as zf:
    members = zf.infolist()
    unsafe = []
    for member in members:
        member_path = Path(member.filename)
        if member_path.is_absolute() or '..' in member_path.parts:
            unsafe.append(member.filename)
    if unsafe:
        raise ValueError(f'Zip có path không an toàn: {unsafe[:5]}')

    names = [member.filename for member in members if not member.is_dir()]
    has_config_dirs = any(name.split('/')[0] in PHASE1_TRAIN_CONFIGS for name in names)
    if not has_config_dirs:
        raise ValueError(
            'Zip không có cấu trúc checkpoint Phase 1 mong đợi. '
            f'Cần thấy thư mục {PHASE1_TRAIN_CONFIGS}; ví dụ file trong zip: {names[:10]}'
        )

    for member in members:
        if member.is_dir():
            continue
        target = phase1_runs_dir / member.filename
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.exists() and not RESTORE_OVERWRITE:
            print(f'Bỏ qua file đã tồn tại: {target}')
            continue
        with zf.open(member) as src, open(target, 'wb') as dst:
            shutil.copyfileobj(src, dst)

print(f'Đã restore checkpoint từ: {RESTORE_ZIP_PATH}')
print(f'Đích: {phase1_runs_dir}')
print('File sau restore:')
for config_id in PHASE1_TRAIN_CONFIGS:
    cfg_dir = phase1_runs_dir / config_id
    print(f'  {config_id}:')
    for filename in ['best.pt', 'last.pt', 'config.json']:
        item = cfg_dir / filename
        status = f'{item.stat().st_size / (1024 * 1024):.1f} MB' if item.exists() else 'missing'
        print(f'    {filename}: {status}')

## 17. Dọn RAM/GPU cache khi chạy lâu

Chạy cell này sau khi interrupt hoặc giữa các lượt train nếu RAM Kaggle phình lên.

In [ ]:
import gc
import os
import subprocess

import torch

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print('Đã gọi gc.collect() và dọn CUDA cache.')
try:
    subprocess.run(['free', '-h'], check=False)
except Exception as exc:
    print(f'Không đọc được RAM bằng free -h: {exc}')

try:
    subprocess.run(['nvidia-smi'], check=False)
except Exception as exc:
    print(f'Không đọc được nvidia-smi: {exc}')